# Predicting Short-Term Stock Market Movements Using Financial News Sentiment Analysis

**CIS 635 – Data Mining and Knowledge Discovery – Fall 2026**
**Team:** Collins Kemboi; [second team member, if applicable]

In this project we look at whether the tone of financial news can help predict short-term stock movements. We collect historical news headlines and daily stock prices, score each headline's sentiment with a language model (FinBERT), and then use data mining methods (logistic regression, decision trees, random forests, support vector machines, and k-means clustering) to see whether the news adds anything beyond what past prices already tell us.

### How this notebook is organized

We follow **CRISP-DM** (Cross-Industry Standard Process for Data Mining). Each phase is a section below, and the table shows where each part of the final report comes from.

| CRISP-DM phase | What we do here | Report section it feeds |
|---|---|---|
| 1. Business Understanding | Define the question, goals and what "success" means | Introduction and motivation |
| 2. Data Understanding | Collect both datasets and explore them | Data description |
| 3. Data Preparation | Clean, match news to trading days, score sentiment, join, build features | Preprocessing and integration |
| 4. Modeling | Train classifiers and run clustering | Methodology |
| 5. Evaluation | Compare models, check against our success criteria | Results, Discussion and limitations |
| 6. Deployment | Save results, explain how to rerun, conclusions | Conclusion, References, Appendix |

### How to run it

1. *(Optional, but faster)* **Runtime → Change runtime type → T4 GPU**. FinBERT runs much faster on a GPU.
2. Choose a `DATA_SOURCE` in the **Settings** cell below.
3. **Runtime → Run all**.

Every code cell has a short note above it explaining what it does and why.

## 0. Setup

### 0.1 Install packages

Colab already comes with pandas, scikit-learn, matplotlib, seaborn, PyTorch and Hugging Face `transformers`. This cell only installs `transformers` if it's missing (for example, if you run the notebook somewhere other than Colab).

In [ ]:
import importlib.util

if importlib.util.find_spec("transformers") is None:
    %pip install -q transformers torch

### 0.2 Import libraries

We group the imports by what they're used for, so it's easy to see which library does what.

In [ ]:
# General tools
import io
import re
import time
import itertools
import warnings
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
import requests

# Charts
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning (scikit-learn)
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score, silhouette_score)

# Statistics (used for a significance test in the Evaluation phase)
from scipy.stats import binomtest

# Display settings: hide noisy warnings, nicer charts, 3 decimals in tables
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)

### 0.3 Settings

All the choices you might want to change are in this one cell.

* `DATA_SOURCE`
  * `"online"`: download news from GDELT and prices from Stooq. The first run is slow because GDELT only allows about one request every 5 seconds. After that, the downloaded files are reused.
  * `"upload"`: use CSV files you upload yourself into `data/raw/` (named `market_<TICKER>.csv` and `news_<TICKER>.csv`).
  * `"synthetic"`: made-up data, **only for checking that the notebook runs**. Never report these numbers.
* `USE_FINBERT`: set to `False` only for a quick test. It switches to a simple word-counting scorer.
* `RANDOM_STATE`: fixes the random seed so we get the same results every time we run the notebook.

In [ ]:
DATA_SOURCE = "synthetic"      # "online", "upload" or "synthetic"

TICKERS = ["AAPL", "MSFT", "AMZN", "TSLA", "NVDA"]
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"

USE_FINBERT = True
STOOQ_API_KEY = None           # only needed if Stooq asks for one
RANDOM_STATE = 42

# Folders for downloaded data and for results (charts, tables)
RAW_DIR = Path("data/raw")
OUTPUT_DIR = Path("outputs")
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
# Phase 1: Business Understanding

### 1.1 Background

Investors react to news: earnings results, lawsuits, product launches, analyst upgrades and downgrades. If the market doesn't fully take in a piece of news on the day it comes out, today's headlines might say something about tomorrow's price move. On the other hand, a lot of news is "priced in" within minutes, so it's not obvious that headlines add anything useful. That's what we want to find out.

### 1.2 Research question

> **Does news sentiment improve next-day directional prediction beyond historical price and volume features?**

### 1.3 Project objectives

1. Link a news dataset and a stock price dataset by company and trading day.
2. Measure the sentiment of every headline with a finance-specific model (FinBERT).
3. Predict whether each stock will close **up or down tomorrow**, using two sets of inputs:
   * **market only:** past prices and trading volume;
   * **market + sentiment:** the same, plus the day's news sentiment.
4. See whether the second version does better on data the models have never seen.
5. Use k-means clustering to explore whether some kinds of days (for example, "very negative news + high volatility") tend to be followed by up or down days.

### 1.4 Data mining goal and success criteria

This is a **binary classification** problem: the label is 1 if tomorrow's close is higher than today's, otherwise 0. We are predicting direction, not the exact price, and we are not building a trading system.

We will call the project's answer "yes, sentiment helps" only if:

* the sentiment models beat the **majority-class baseline** (always guessing the most common direction), **and**
* adding sentiment improves test accuracy / ROC-AUC for **most** of the four models, not just one, **and**
* the improvement is bigger than what random chance could explain (we check this with McNemar's test in Phase 5).

### 1.5 Project plan and risks

| Week | Work |
|---|---|
| 1 | Confirm sources, download and join the datasets |
| 2 | Explore and clean the data |
| 3 | Sentiment scoring and feature engineering |
| 4 | Market-only baseline models |
| 5 | Sentiment-enhanced models |
| 6 | Time-aware tuning and held-out evaluation |
| 7 | Report and presentation |
| 8 | Buffer, reproducibility checks, finalize |

**Main risks** and how we handle them: limited or noisy news coverage (company-name filtering, duplicate removal); timestamps that don't line up with trading hours (careful mapping to trading sessions in Phase 3); class imbalance (balanced class weights and several metrics); overfitting and **look-ahead leakage**, which means accidentally using future information (chronological splits and features that only use past data).

---
# Phase 2: Data Understanding

### 2.1 The two datasets and how they are related

| | Dataset 1: Financial news | Dataset 2: Stock market data |
|---|---|---|
| Source | GDELT DOC 2.0 API (gdeltproject.org) | Stooq (stooq.com) |
| One row is | one headline: title, URL, website, time in UTC | one trading day for one stock: open, high, low, close, volume |
| Coverage | English-language web news, 2017 onward | daily history of U.S. stocks |
| Terms | free for research/academic use with citation | free download; check the site's terms before use |

**How they connect:** both datasets share the **stock ticker** and **time**. In Phase 3 we assign every headline to the trading day it can affect, summarize the sentiment for each stock and day, and join it to the price table on *ticker + date*. The prices give us the label (up or down tomorrow) and the news gives us the sentiment features, so we need **both** datasets to answer the question.

### 2.2 The companies we study

We picked five large companies that are in the news almost every day. For each one we store:

* `query`: the search terms we send to GDELT;
* `pattern`: a regular expression the headline must match to be kept. This filters out things like "apple pie recipe" showing up as Apple news.

In [ ]:
COMPANIES = {
    "AAPL": {"name": "Apple",     "query": '("Apple Inc" OR "Apple" OR "iPhone" OR AAPL)', "pattern": r"\b(?:apple|iphone|aapl|tim cook)\b"},
    "MSFT": {"name": "Microsoft", "query": '("Microsoft" OR MSFT)', "pattern": r"\b(?:microsoft|msft|satya nadella)\b"},
    "AMZN": {"name": "Amazon",    "query": '("Amazon" OR AMZN)',    "pattern": r"\b(?:amazon|amzn|aws|andy jassy)\b"},
    "TSLA": {"name": "Tesla",     "query": '("Tesla" OR TSLA)',     "pattern": r"\b(?:tesla|tsla|elon musk)\b"},
    "NVDA": {"name": "Nvidia",    "query": '("Nvidia" OR NVDA)',    "pattern": r"\b(?:nvidia|nvda|jensen huang)\b"},
}

### 2.3 Function to download stock prices (Stooq)

Stooq gives a stock's daily history as a CSV file at a simple URL. U.S. tickers need `.us` at the end (e.g. `aapl.us`).

If the reply doesn't start with the `Date` header, it isn't real data (usually an error page or a request for an API key), so we stop with a clear message instead of saving junk.

In [ ]:
def download_prices_from_stooq(ticker, start, end, api_key=None):
    params = {
        "s": ticker.lower() + ".us",                    # e.g. "aapl.us"
        "i": "d",                                       # daily data
        "d1": pd.Timestamp(start).strftime("%Y%m%d"),   # start date as YYYYMMDD
        "d2": pd.Timestamp(end).strftime("%Y%m%d"),     # end date as YYYYMMDD
    }
    if api_key:
        params["apikey"] = api_key

    response = requests.get("https://stooq.com/q/d/l/", params=params, timeout=30)
    response.raise_for_status()

    if not response.text.lower().startswith("date"):
        raise RuntimeError(f"Stooq didn't return price data for {ticker}: {response.text[:200]}")

    return pd.read_csv(io.StringIO(response.text))

### 2.4 Function to ask GDELT for headlines

This sends **one** request to the GDELT DOC API: "give me English articles matching this query between these two times."

* `mode=artlist` asks for a list of articles. Each one has a title, URL, website and `seendate`, which is when GDELT found the article, in UTC.
* GDELT returns at most **250 articles per request**.
* If something goes wrong (for example, we ask too fast), GDELT replies with plain text instead of JSON. We check for that.

In [ ]:
def request_gdelt(query, start, end):
    params = {
        "query": query + " sourcelang:english",
        "mode": "artlist",
        "format": "json",
        "maxrecords": 250,
        "sort": "datedesc",
        "startdatetime": start.strftime("%Y%m%d%H%M%S"),
        "enddatetime": end.strftime("%Y%m%d%H%M%S"),
    }
    response = requests.get("https://api.gdeltproject.org/api/v2/doc/doc", params=params, timeout=60)
    response.raise_for_status()

    if not response.text.strip().startswith("{"):
        raise RuntimeError("GDELT error: " + response.text[:200])

    return response.json().get("articles", [])

### 2.5 Function to download all headlines for one company

Because of the 250-article limit, we walk through the date range **3 days at a time**. Things to note:

* We wait about 5.5 seconds between requests, as GDELT asks.
* If a request fails, we try again up to 2 more times, waiting a bit longer each time.
* At the end we keep just the four columns we need, and convert GDELT's time format (`20240102T143000Z`) into a proper UTC timestamp.

In [ ]:
def download_headlines_from_gdelt(ticker, start, end, days_per_request=3, wait_seconds=5.5):
    articles = []
    window_start = pd.Timestamp(start)
    last_day = pd.Timestamp(end) + pd.Timedelta(days=1)

    while window_start < last_day:
        window_end = min(window_start + pd.Timedelta(days=days_per_request), last_day)

        for attempt in range(3):
            try:
                articles += request_gdelt(COMPANIES[ticker]["query"], window_start, window_end)
                break
            except (requests.RequestException, RuntimeError, ValueError):
                if attempt == 2:
                    raise                                     # give up after 3 tries
                time.sleep(wait_seconds * (attempt + 2))      # wait longer, then retry

        window_start = window_end
        time.sleep(wait_seconds)

    if not articles:
        return pd.DataFrame(columns=["published_utc", "headline", "url", "source"])

    raw = pd.DataFrame(articles)
    return pd.DataFrame({
        "published_utc": pd.to_datetime(raw["seendate"], format="%Y%m%dT%H%M%SZ", utc=True),
        "headline": raw["title"],
        "url": raw["url"],
        "source": raw["domain"],
    })

### 2.6 Synthetic data (for testing only)

When `DATA_SOURCE = "synthetic"`, we make up prices and headlines so the whole notebook can run without internet access.

How the fake data works:

1. Every day gets a hidden "news tone" (a random number; positive = good news).
2. Headlines for that day are more likely to be positive when the tone is positive.
3. **Tomorrow's** return gets a small push in the direction of **today's** tone.

So there *is* a signal for the models to find. That's useful for checking the code, but **the results mean nothing about the real market.**

In [ ]:
FAKE_HEADLINES = {
    "positive": ["{} shares surge after earnings beat", "{} gains on strong growth outlook", "Analysts upgrade {} on record profit"],
    "negative": ["{} shares plunge after earnings miss", "{} falls on weak outlook", "Analysts downgrade {} amid probe"],
    "neutral":  ["{} to hold annual shareholder meeting", "{} names new board member", "What to watch for {} this week"],
}


def make_fake_prices(days, tone, rng, signal=0.006):
    returns = rng.normal(0.0003, 0.018, len(days))   # normal day-to-day noise
    returns[1:] += signal * tone[:-1]                 # yesterday's news nudges today's return
    close = 100 * np.exp(np.cumsum(returns))
    open_ = close * np.exp(rng.normal(0, 0.005, len(days)))
    return pd.DataFrame({
        "Date": days,
        "Open": open_,
        "High": np.maximum(open_, close) * (1 + abs(rng.normal(0, 0.008, len(days)))),
        "Low": np.minimum(open_, close) * (1 - abs(rng.normal(0, 0.008, len(days)))),
        "Close": close,
        "Volume": (rng.lognormal(16, 0.3, len(days)) * (1 + 0.3 * abs(tone))).round(),
    })


def make_fake_headlines(name, days, tone, rng):
    rows = []
    previous_close = (days[0] - pd.Timedelta(hours=8)).tz_localize("America/New_York")
    for day, day_tone in zip(days, tone):
        close_time = (day + pd.Timedelta(hours=16)).tz_localize("America/New_York")
        chance_positive = 1 / (1 + np.exp(-1.5 * day_tone))
        n_headlines = rng.poisson(3 * (1 + abs(day_tone)))      # big news days have more headlines

        for _ in range(n_headlines):
            if rng.random() < 0.35:
                kind = "neutral"
            elif rng.random() < chance_positive:
                kind = "positive"
            else:
                kind = "negative"
            # publish at a random time between the previous close and this day's close
            published = previous_close + (close_time - previous_close) * rng.uniform(0, 0.999)
            rows.append({"published_utc": published.tz_convert("UTC"),
                         "headline": rng.choice(FAKE_HEADLINES[kind]).format(name),
                         "url": None, "source": "synthetic"})
        previous_close = close_time

    headlines = pd.DataFrame(rows)
    # Repost 5% of the headlines, like real news sites do, so the cleaning step has something to remove
    return pd.concat([headlines, headlines.sample(frac=0.05, random_state=0)], ignore_index=True)


def make_synthetic_data(tickers, start, end, seed=0):
    rng = np.random.default_rng(seed)
    days = pd.bdate_range(start, end)                                         # weekdays only
    days = days[[(d.month, d.day) not in [(1, 1), (7, 4), (12, 25)] for d in days]]  # drop a few holidays
    prices, headlines = {}, {}
    for ticker in tickers:
        tone = rng.normal(0, 1, len(days))
        prices[ticker] = make_fake_prices(days, tone, rng)
        headlines[ticker] = make_fake_headlines(COMPANIES[ticker]["name"], days, tone, rng)
    return prices, headlines

### 2.7 Load both datasets

For each company we:

1. use the saved CSV file in `data/raw/` if we already have it, so we never download the same thing twice;
2. otherwise download it (in `"online"` mode) and save it;
3. or stop with a message if the file is missing in `"upload"` mode.

We keep the raw data in two dictionaries, `raw_prices` and `raw_news`, with one table per ticker.

In [ ]:
raw_prices, raw_news = {}, {}

if DATA_SOURCE == "synthetic":
    raw_prices, raw_news = make_synthetic_data(TICKERS, START_DATE, END_DATE, seed=RANDOM_STATE)

else:
    for ticker in TICKERS:
        price_file = RAW_DIR / f"market_{ticker}.csv"
        news_file = RAW_DIR / f"news_{ticker}.csv"

        if not price_file.exists():
            if DATA_SOURCE == "upload":
                raise FileNotFoundError(f"Please upload {price_file}")
            prices_downloaded = download_prices_from_stooq(ticker, START_DATE, END_DATE, STOOQ_API_KEY)
            prices_downloaded.to_csv(price_file, index=False)

        if not news_file.exists():
            if DATA_SOURCE == "upload":
                raise FileNotFoundError(f"Please upload {news_file}")
            print(f"Downloading {ticker} headlines from GDELT (this takes a while)...")
            download_headlines_from_gdelt(ticker, START_DATE, END_DATE).to_csv(news_file, index=False)

        raw_prices[ticker] = pd.read_csv(price_file)
        raw_news[ticker] = pd.read_csv(news_file)

print("Loaded data for:", ", ".join(raw_prices))

### 2.8 Dataset sizes

A quick count of rows per company in each dataset, before any cleaning.

In [ ]:
sizes = pd.DataFrame({
    "price rows": {t: len(raw_prices[t]) for t in TICKERS},
    "headlines (raw)": {t: len(raw_news[t]) for t in TICKERS},
})
sizes.loc["TOTAL"] = sizes.sum()
sizes

### 2.9 First look at the price data

`head()` shows the first rows so we can see the columns. `describe()` gives summary statistics, which helps spot problems like negative prices or zero volume.

In [ ]:
example_ticker = TICKERS[0]
display(raw_prices[example_ticker].head())
raw_prices[example_ticker].describe()

### 2.10 First look at the headlines

What a headline row looks like, and the time span of the news we got for each company.

In [ ]:
display(raw_news[example_ticker].head())

news_span = pd.DataFrame({
    t: {"first headline": pd.to_datetime(raw_news[t]["published_utc"], utc=True).min(),
        "last headline": pd.to_datetime(raw_news[t]["published_utc"], utc=True).max()}
    for t in TICKERS
}).T
news_span

### 2.11 Data quality check

Before cleaning, we count the problems we'll have to fix in Phase 3:

* **missing values** in important columns;
* **duplicate dates** in the price data;
* **duplicate headlines** (the same story posted by several websites).

In [ ]:
quality = []
for t in TICKERS:
    p = raw_prices[t].rename(columns=str.lower)
    n = raw_news[t]
    quality.append({
        "ticker": t,
        "missing close prices": p["close"].isna().sum(),
        "duplicate price dates": p["date"].duplicated().sum(),
        "missing headlines": n["headline"].isna().sum(),
        "repeated headline text": n["headline"].str.lower().duplicated().sum(),
    })
pd.DataFrame(quality).set_index("ticker")

### 2.12 Closing prices over time

A chart of each stock's closing price. We want to check that the data has no obvious gaps or strange jumps, and see what kind of market period we're studying (rising, falling, sideways).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for t in TICKERS:
    p = raw_prices[t].rename(columns=str.lower)
    ax.plot(pd.to_datetime(p["date"]), p["close"], label=t, linewidth=1)
ax.set_title("Daily closing price")
ax.set_ylabel("price ($)")
ax.legend()
plt.show()

### 2.13 When are headlines published?

Headlines come out at all hours, but the market is only open 9:30 am to 4:00 pm New York time on weekdays. This chart shows how many headlines are published at each hour (New York time). Everything to the right of the red line comes out **after the market has closed**, so it can only affect the **next** trading day. This is why we need the session-matching step in Phase 3.

In [ ]:
all_news_times = pd.concat([pd.to_datetime(raw_news[t]["published_utc"], utc=True) for t in TICKERS])
ny_hour = all_news_times.dt.tz_convert("America/New_York").dt.hour

fig, ax = plt.subplots(figsize=(10, 3.5))
ny_hour.value_counts().sort_index().plot.bar(ax=ax, color="steelblue", label="headlines")
ax.axvline(16 - 0.5, color="red", linestyle="--", label="4:00 pm close")
ax.set_xlabel("hour of day (New York time)")
ax.set_ylabel("number of headlines")
ax.set_title("Headlines by publishing hour")
ax.legend()
plt.show()

weekend = all_news_times.dt.tz_convert("America/New_York").dt.dayofweek >= 5
print(f"{(ny_hour >= 16).mean():.1%} of headlines come out after the close, {weekend.mean():.1%} on weekends.")

---
# Phase 3: Data Preparation

In this phase we turn the two raw datasets into **one table** with one row per *(stock, trading day)*, holding the features and the label. The steps are:

1. clean the prices;
2. clean the headlines;
3. match each headline to a trading day;
4. score each headline's sentiment with FinBERT;
5. summarize sentiment per stock per day and **join** it to the prices;
6. build the features and the label;
7. split the data by time into train / validation / test.

### 3.1 Clean the price data

What this function does:

* makes column names lowercase (`Close` → `close`) so the code works however the file was formatted;
* turns the `date` column into real dates and the price columns into numbers (anything that can't be converted becomes missing);
* removes days with no closing price, repeated dates, and impossible values (price ≤ 0 or negative volume);
* keeps only our study period and adds a `ticker` column so we can stack all companies into one table.

In [ ]:
def clean_prices(prices, ticker):
    prices = prices.rename(columns=lambda c: c.strip().lower())
    prices = prices[["date", "open", "high", "low", "close", "volume"]].copy()

    prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
    for col in ["open", "high", "low", "close", "volume"]:
        prices[col] = pd.to_numeric(prices[col], errors="coerce")

    prices = prices.dropna(subset=["close"])
    prices = prices.drop_duplicates(subset="date", keep="last")
    prices = prices[(prices["close"] > 0) & (prices["volume"].fillna(0) >= 0)]
    prices = prices[(prices["date"] >= START_DATE) & (prices["date"] <= END_DATE)]

    prices.insert(0, "ticker", ticker)
    return prices.sort_values("date").reset_index(drop=True)

Now we run it for every company and stack the results into one `prices` table. The summary shows the first and last date and the number of trading days per stock.

In [ ]:
prices = pd.concat([clean_prices(raw_prices[t], t) for t in TICKERS], ignore_index=True)
prices.groupby("ticker")["date"].agg(["min", "max", "count"])

### 3.2 Clean the headlines

We fix the three problems found in Phase 2:

1. **Missing values:** drop headlines without a title or timestamp.
2. **Wrong company:** keep a headline only if it matches the company's `pattern` from 2.2.
3. **Duplicates:** the same story often appears on many websites. We make a simplified copy of each headline (lowercase, no punctuation) and keep only the earliest copy per day.

The function also counts how many rows each step removed, so we can report that.

In [ ]:
def clean_headlines(news, ticker):
    news = news.copy()
    news["published_utc"] = pd.to_datetime(news["published_utc"], utc=True)
    news["headline"] = news["headline"].astype("string").str.strip()
    counts = {"raw": len(news)}

    # 1. missing values
    news = news.dropna(subset=["published_utc", "headline"])
    news = news[news["headline"].str.len() > 0]
    counts["missing removed"] = counts["raw"] - len(news)

    # 2. headlines that aren't about this company
    about_company = news["headline"].str.contains(COMPANIES[ticker]["pattern"], flags=re.IGNORECASE, regex=True)
    counts["off-topic removed"] = int((~about_company).sum())
    news = news[about_company]

    # 3. same story reposted on the same day
    news["simple_text"] = (news["headline"].str.lower()
                           .str.replace(r"[^a-z0-9 ]", "", regex=True)
                           .str.split().str.join(" "))
    news["day"] = news["published_utc"].dt.floor("D")
    before = len(news)
    news = news.sort_values("published_utc").drop_duplicates(subset=["simple_text", "day"])
    counts["duplicates removed"] = before - len(news)

    news = news.drop(columns=["simple_text", "day"])
    news.insert(0, "ticker", ticker)
    counts["kept"] = len(news)
    return news.reset_index(drop=True), counts

Run the cleaning for every company. The table shows how many headlines each step removed, which is useful for the "Data preprocessing" part of the report.

In [ ]:
cleaned_list, cleaning_counts = [], {}
for t in TICKERS:
    cleaned, cleaning_counts[t] = clean_headlines(raw_news[t], t)
    cleaned_list.append(cleaned)

headlines = pd.concat(cleaned_list, ignore_index=True)
pd.DataFrame(cleaning_counts).T

### 3.3 Match each headline to a trading day

This step links the two datasets, and it's where it's easiest to make a mistake.

**The rule.** We pretend we make each prediction at the market close (4:00 pm New York time) on day *t*, predicting whether the stock closes higher on day *t+1*. A headline can only be used for day *t* if it was published **before** day *t*'s close:

| Headline published… | Counts toward… |
|---|---|
| before 4:00 pm on a trading day | that same day |
| after 4:00 pm | the next trading day |
| on a weekend or holiday | the next trading day |

**How the code does it:**

1. Convert the UTC time to New York time. This also handles daylight saving time.
2. If it's 4:00 pm or later, move the date forward one day.
3. Find the first **trading day** on or after that date. `np.searchsorted` does this quickly. We get the list of trading days from the price data, so weekends and holidays are skipped automatically.
4. Headlines after the last trading day in our data have nowhere to go, so we drop them.

In [ ]:
def match_to_trading_day(news, prices):
    matched = []
    for ticker, group in news.groupby("ticker"):
        trading_days = np.sort(prices.loc[prices["ticker"] == ticker, "date"].unique())

        ny_time = group["published_utc"].dt.tz_convert("America/New_York")      # step 1
        calendar_day = ny_time.dt.tz_localize(None).dt.normalize()
        after_close = ny_time.dt.hour >= 16
        earliest_day = calendar_day.where(~after_close, calendar_day + pd.Timedelta(days=1))  # step 2

        position = np.searchsorted(trading_days, earliest_day.to_numpy(dtype="datetime64[ns]"))  # step 3
        has_trading_day = position < len(trading_days)                         # step 4

        group = group[has_trading_day].copy()
        group["session"] = pd.to_datetime(trading_days[position[has_trading_day]])
        matched.append(group)
    return pd.concat(matched, ignore_index=True)


headlines = match_to_trading_day(headlines, prices)
print(f"{len(headlines):,} headlines matched to a trading day")

**Checking that the matching works.** We print a few random headlines with their New York publishing time and the trading day they were assigned to. A headline from Saturday should land on Monday, and one from 6 pm Tuesday should land on Wednesday.

In [ ]:
check = headlines.sample(10, random_state=1)[["ticker", "published_utc", "session", "headline"]].copy()
check["published (New York)"] = check["published_utc"].dt.tz_convert("America/New_York").dt.strftime("%a %Y-%m-%d %H:%M")
check["assigned trading day"] = check["session"].dt.strftime("%a %Y-%m-%d")
check[["ticker", "published (New York)", "assigned trading day", "headline"]]

### 3.4 Score sentiment with FinBERT

**What FinBERT is.** FinBERT (`ProsusAI/finbert`) is a BERT language model that was further trained on financial text. For each headline it gives three probabilities that add up to 1: **positive**, **negative** and **neutral**.

**Why not a general sentiment tool?** Words mean different things in finance. "Liability", "volatile" or "short" are neutral or technical in a financial headline but would look negative to a general-purpose tool.

#### 3.4.1 Load the model

The first time, this downloads the model (about 400 MB) from Hugging Face. We use the GPU if there is one.

In [ ]:
LABELS = ["positive", "negative", "neutral"]


def load_finbert():
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(device).eval()
    print("FinBERT loaded on", device)
    return tokenizer, model, device

#### 3.4.2 Score headlines in batches

Scoring headlines 32 at a time is much faster than one by one. For each batch:

1. the **tokenizer** turns text into numbers the model understands (headlines are short, so we cut them at 64 tokens);
2. the model outputs a raw score per class, and **softmax** turns those into probabilities;
3. we reorder the columns to always be *[positive, negative, neutral]*, because the model's own order might be different.

`torch.no_grad()` tells PyTorch we're only predicting, not training, which saves memory and time.

In [ ]:
def finbert_scores(texts, tokenizer, model, device, batch_size=32):
    import torch

    model_labels = {i: label.lower() for i, label in model.config.id2label.items()}
    column_order = [next(i for i, l in model_labels.items() if l == wanted) for wanted in LABELS]

    all_probs = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            tokens = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            probs = torch.softmax(model(**tokens).logits, dim=-1).cpu().numpy()
            all_probs.append(probs[:, column_order])
            if (start // batch_size) % 100 == 0:
                print(f"  scored {start + len(batch):,} of {len(texts):,} headlines")
    return np.vstack(all_probs)

#### 3.4.3 Backup scorer (for testing only)

If FinBERT can't be loaded (no internet, no `transformers`), we fall back to counting positive and negative finance words in the headline. This lets the notebook run anywhere, but the results are much cruder, so **final results must use FinBERT**.

In [ ]:
POSITIVE_WORDS = {"beat", "beats", "surge", "surges", "gain", "gains", "jump", "record", "growth", "upgrade",
                  "strong", "profit", "rise", "rises", "rally", "boost", "raises", "higher", "tops"}
NEGATIVE_WORDS = {"miss", "misses", "plunge", "plunges", "fall", "falls", "drop", "slump", "loss", "downgrade",
                  "weak", "lawsuit", "probe", "recall", "cut", "cuts", "decline", "layoffs", "warns", "lower"}


def word_list_scores(texts):
    scores = []
    for text in texts:
        words = re.findall(r"[a-z]+", text.lower())
        n_positive = sum(w in POSITIVE_WORDS for w in words)
        n_negative = sum(w in NEGATIVE_WORDS for w in words)
        raw = np.array([n_positive, n_negative, 0.5])      # 0.5 = slight lean toward neutral
        scores.append(np.exp(raw) / np.exp(raw).sum())    # softmax -> probabilities
    return np.array(scores)

#### 3.4.4 Score every headline

We try FinBERT first and fall back to the word list only if it fails. The three probabilities are added as columns `p_pos`, `p_neg`, `p_neu`, and the label with the highest probability goes in `sentiment`.

In [ ]:
texts = headlines["headline"].astype(str).tolist()
scorer_used = "word list (fallback)"

if USE_FINBERT:
    try:
        tokenizer, finbert, device = load_finbert()
        probs = finbert_scores(texts, tokenizer, finbert, device)
        scorer_used = "FinBERT"
    except Exception as error:
        print("Couldn't use FinBERT, falling back to the word list:", error)
        probs = word_list_scores(texts)
else:
    probs = word_list_scores(texts)

headlines["p_pos"] = probs[:, 0]
headlines["p_neg"] = probs[:, 1]
headlines["p_neu"] = probs[:, 2]
headlines["sentiment"] = [LABELS[i] for i in probs.argmax(axis=1)]
headlines["net_sentiment"] = headlines["p_pos"] - headlines["p_neg"]   # > 0 means more positive than negative

print("Scorer used:", scorer_used)
headlines["sentiment"].value_counts(normalize=True).rename("share of headlines")

#### 3.4.5 Do the scores make sense?

A quick manual check: the 5 most positive and 5 most negative headlines. If these look wrong, something is off with the scoring.

In [ ]:
most_positive = headlines.nlargest(5, "net_sentiment")
most_negative = headlines.nsmallest(5, "net_sentiment")
pd.concat([most_positive, most_negative])[["ticker", "headline", "p_pos", "p_neg", "p_neu"]]

#### 3.4.6 Sentiment mix per company

The share of positive, negative and neutral headlines for each company. Companies with a lot of negative coverage in this period should show it here.

In [ ]:
mix = pd.crosstab(headlines["ticker"], headlines["sentiment"], normalize="index")[LABELS]
mix.plot.bar(stacked=True, figsize=(8, 3.5), color=["seagreen", "indianred", "lightgrey"],
             title="Sentiment mix of headlines per company")
plt.ylabel("share of headlines")
plt.legend(loc="upper right", fontsize=8)
plt.show()

### 3.5 Daily sentiment and joining the two datasets

#### 3.5.1 Summarize sentiment per stock per trading day

A stock can have many headlines on the same day, so we combine them into one row per *(ticker, trading day)*:

* `news_count`: how many headlines;
* `sent_pos_mean`, `sent_neg_mean`, `sent_neu_mean`: average probabilities;
* `sent_net_mean`: average of (positive − negative);
* `sent_net_std`: how much the headlines disagree (0 when there's only one headline);
* `sent_pos_share`, `sent_neg_share`: share of headlines labelled positive / negative.

In [ ]:
headlines["is_positive"] = (headlines["sentiment"] == "positive").astype(float)
headlines["is_negative"] = (headlines["sentiment"] == "negative").astype(float)

daily_sentiment = headlines.groupby(["ticker", "session"]).agg(
    news_count=("headline", "size"),
    sent_pos_mean=("p_pos", "mean"),
    sent_neg_mean=("p_neg", "mean"),
    sent_neu_mean=("p_neu", "mean"),
    sent_net_mean=("net_sentiment", "mean"),
    sent_net_std=("net_sentiment", "std"),
    sent_pos_share=("is_positive", "mean"),
    sent_neg_share=("is_negative", "mean"),
).reset_index()

daily_sentiment = daily_sentiment.rename(columns={"session": "date"})
daily_sentiment["sent_net_std"] = daily_sentiment["sent_net_std"].fillna(0)
daily_sentiment.head()

#### 3.5.2 Join news and prices

This is the **join** between our two datasets, on `ticker` and `date`.

We use a **left join** (keep every row of the price table) so that trading days with **no news** stay in the data. For those days we set the counts and sentiment values to 0. The `has_news` feature (added in 3.6) tells the model which days these are.

In [ ]:
data = prices.merge(daily_sentiment, on=["ticker", "date"], how="left")

sentiment_columns = [c for c in daily_sentiment.columns if c not in ("ticker", "date")]
data[sentiment_columns] = data[sentiment_columns].fillna(0)

print(f"{len(data):,} stock-days after the join")
print(f"{(data['news_count'] > 0).mean():.1%} of stock-days have at least one headline")
data.head()

### 3.6 Feature engineering and the label

**The most important rule:** every feature for day *t* may only use information we'd have at **day t's close**. All rolling calculations below only look backwards, so this holds.

#### 3.6.1 Market features (from prices and volume)

| Feature | What it means |
|---|---|
| `ret_1` … `ret_5` | log return today, yesterday, … 4 days ago (short-term momentum) |
| `ret_5d_cum` | total return over the last 5 days |
| `vol_5`, `vol_20` | volatility: how much returns jumped around over 5 and 20 days |
| `volume_chg` | change in trading volume from yesterday |
| `volume_z20` | how unusual today's volume is vs. the last 20 days (z-score) |
| `hl_range` | (high − low) / close: how much the price moved within the day |
| `close_sma10_ratio` | how far the close is above/below its 10-day average |

We use **log returns**, `log(today / yesterday)`, because they add up nicely over several days.

In [ ]:
def add_market_features(stock):
    stock = stock.sort_values("date").copy()
    log_return = np.log(stock["close"]).diff()

    for k in range(1, 6):
        stock[f"ret_{k}"] = log_return.shift(k - 1)        # ret_1 = today, ret_2 = yesterday, ...
    stock["ret_5d_cum"] = log_return.rolling(5).sum()
    stock["vol_5"] = log_return.rolling(5).std()
    stock["vol_20"] = log_return.rolling(20).std()

    log_volume = np.log1p(stock["volume"])
    stock["volume_chg"] = log_volume.diff()
    stock["volume_z20"] = (log_volume - log_volume.rolling(20).mean()) / log_volume.rolling(20).std()

    stock["hl_range"] = (stock["high"] - stock["low"]) / stock["close"]
    stock["close_sma10_ratio"] = stock["close"] / stock["close"].rolling(10).mean() - 1
    return stock

#### 3.6.2 Sentiment features

On top of the daily averages from 3.5.1 we add:

* `has_news`: 1 if there was at least one headline that day, else 0;
* `news_count_log`: log of the number of headlines (so a day with 200 headlines doesn't dwarf everything else);
* `sent_net_mean_lag1`: **yesterday's** net sentiment (news might take more than a day to sink in);
* `sent_net_3d` and `news_count_3d_log`: 3-day rolling average sentiment and headline count.

In [ ]:
def add_sentiment_features(stock):
    stock = stock.sort_values("date").copy()
    stock["has_news"] = (stock["news_count"] > 0).astype(float)
    stock["news_count_log"] = np.log1p(stock["news_count"])
    stock["sent_net_mean_lag1"] = stock["sent_net_mean"].shift(1).fillna(0)
    stock["sent_net_3d"] = stock["sent_net_mean"].rolling(3, min_periods=1).mean()
    stock["news_count_3d_log"] = np.log1p(stock["news_count"].rolling(3, min_periods=1).sum())
    return stock

#### 3.6.3 The label (what we predict)

`target = 1` if **tomorrow's** close is higher than today's, otherwise `0`.

The last day of each stock has no "tomorrow" in our data, so its label is left empty (NaN) and that row is removed later.

In [ ]:
def add_label(stock):
    stock = stock.sort_values("date").copy()
    stock["next_close"] = stock["close"].shift(-1)                      # tomorrow's close
    stock["target"] = (stock["next_close"] > stock["close"]).astype(float)
    stock.loc[stock["next_close"].isna(), "target"] = np.nan           # no tomorrow -> no label
    return stock

#### 3.6.4 Build the final modeling table

We apply the three functions **one stock at a time** (`groupby("ticker")`), so that one company's prices never leak into another company's rolling averages.

Then we remove rows that can't be used:

* the first ~20 days of each stock, which don't have enough history for the 20-day features;
* the last day of each stock, which has no label.

In [ ]:
MARKET_FEATURES = ["ret_1", "ret_2", "ret_3", "ret_4", "ret_5", "ret_5d_cum", "vol_5", "vol_20",
                   "volume_chg", "volume_z20", "hl_range", "close_sma10_ratio"]

SENTIMENT_FEATURES = ["has_news", "news_count_log", "sent_pos_mean", "sent_neg_mean", "sent_neu_mean",
                      "sent_net_mean", "sent_net_std", "sent_pos_share", "sent_neg_share",
                      "sent_net_mean_lag1", "sent_net_3d", "news_count_3d_log"]

stocks = []
for ticker, stock in data.groupby("ticker"):
    stock = add_market_features(stock)
    stock = add_sentiment_features(stock)
    stock = add_label(stock)
    stocks.append(stock)

data = pd.concat(stocks, ignore_index=True)
data = data.dropna(subset=MARKET_FEATURES + ["target"])
data["target"] = data["target"].astype(int)
data = data.sort_values(["date", "ticker"]).reset_index(drop=True)

print(f"Modeling table: {len(data):,} rows")
print(f"{len(MARKET_FEATURES)} market features + {len(SENTIMENT_FEATURES)} sentiment features")

### 3.7 Explore the prepared data

#### 3.7.1 Class balance

How often does a stock go up the next day? If it's far from 50%, accuracy alone would be misleading. A model could look good by always guessing the common class. That's why we also report balanced accuracy and F1.

In [ ]:
balance = data.groupby("ticker")["target"].mean()
ax = balance.plot.bar(figsize=(7, 3.5), color="steelblue", title="Share of days followed by an up day")
ax.axhline(0.5, linestyle="--", color="grey")
ax.set_ylabel("share of up days")
plt.show()
print(f"Overall: {data['target'].mean():.1%} up days")

#### 3.7.2 Does sentiment line up with next-day returns?

We sort the days that had news into 5 equal groups (quintiles), from most negative to most positive sentiment, and plot the average next-day return of each group. If sentiment carries information, the bars should rise from left to right. This is only a first look; the models in Phase 4 test it properly.

In [ ]:
news_days = data[data["has_news"] == 1].copy()
news_days["next_day_return"] = np.log(news_days["next_close"] / news_days["close"])
news_days["sentiment_group"] = pd.qcut(news_days["sent_net_mean"], 5, duplicates="drop",
                                       labels=["most negative", "2", "3", "4", "most positive"])

ax = news_days.groupby("sentiment_group", observed=True)["next_day_return"].mean().plot.bar(
    figsize=(7, 3.5), color="steelblue", title="Average next-day return by sentiment group")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("mean log return")
plt.xticks(rotation=0)
plt.savefig(OUTPUT_DIR / "sentiment_vs_return.png", dpi=130, bbox_inches="tight")
plt.show()

#### 3.7.3 Correlation with the label

The correlation of each sentiment feature with the next-day label. On real stock data, expect small numbers (well under 0.1). Markets are noisy, and even a small, consistent edge can matter.

In [ ]:
correlations = data[SENTIMENT_FEATURES + ["target"]].corr()["target"].drop("target").sort_values()
correlations.plot.barh(figsize=(6, 4), color="steelblue", title="Correlation with next-day up/down")
plt.show()

### 3.8 Split the data by time

Stock data is a **time series**, so we must **not** shuffle it. Training on 2024 and testing on 2022 would let the model "see the future." We split by date instead:

| Part | Share of trading days | Used for |
|---|---|---|
| Training | first 60% | fitting the models |
| Validation | next 20% | choosing model settings |
| Test | last 20% | the final, one-time evaluation |

We also leave a **one-day gap** after the training and validation periods. The label of the last training day uses the *next* day's close, which belongs to the validation period, so without the gap a little information would leak across.

In [ ]:
all_dates = np.sort(data["date"].unique())
n_dates = len(all_dates)
train_end = int(n_dates * 0.6)
val_end = int(n_dates * 0.8)
GAP = 1

train = data[data["date"].isin(all_dates[:train_end - GAP])]
val = data[data["date"].isin(all_dates[train_end:val_end - GAP])]
test = data[data["date"].isin(all_dates[val_end:])]
train_val = pd.concat([train, val])      # used to retrain the final models

split_summary = pd.DataFrame({
    name: {"from": part["date"].min().date(), "to": part["date"].max().date(),
           "rows": len(part), "up days": f"{part['target'].mean():.1%}"}
    for name, part in [("train", train), ("validation", val), ("test", test)]
}).T
split_summary

A picture of the split on a timeline, using one stock's price. This is a nice figure for the presentation.

In [ ]:
one_stock = data[data["ticker"] == TICKERS[0]]
fig, ax = plt.subplots(figsize=(12, 3.5))
for part, color, name in [(train, "tab:blue", "train"), (val, "tab:orange", "validation"), (test, "tab:green", "test")]:
    rows = one_stock[one_stock["date"].isin(part["date"])]
    ax.plot(rows["date"], rows["close"], color=color, label=name)
ax.set_title(f"Chronological split ({TICKERS[0]} closing price)")
ax.legend()
plt.savefig(OUTPUT_DIR / "time_split.png", dpi=130, bbox_inches="tight")
plt.show()

---
# Phase 4: Modeling

### 4.1 The two feature sets

Every model is trained twice, once with each feature set. Comparing them answers our research question.

In [ ]:
FEATURE_SETS = {
    "market_only": MARKET_FEATURES,
    "market_plus_sentiment": MARKET_FEATURES + SENTIMENT_FEATURES,
}
for name, columns in FEATURE_SETS.items():
    print(f"{name}: {len(columns)} features")

### 4.2 The models and the settings we try

| Model | Why we use it |
|---|---|
| **Logistic regression** | Simple linear baseline; easy to interpret |
| **Decision tree** | Learns if-then rules, e.g. "negative news *and* high volatility → down" |
| **Random forest** | Many trees averaged together; usually more accurate and stable than one tree |
| **Support vector machine** (RBF kernel) | Can learn curved (non-linear) boundaries between up and down days |

Each model comes with a small **grid** of settings to try:

* `C` (logistic regression, SVM): how strongly we hold the model back from overfitting. Smaller means simpler.
* `max_depth`, `min_samples_leaf` (trees): how big the trees are allowed to grow.
* `class_weight="balanced"`: gives the less common class more weight, in case up and down days aren't 50/50.

We keep the grids small on purpose. Trying hundreds of settings would make it easy to find one that looks good on the validation set just by luck.

In [ ]:
MODELS = {
    "logistic_regression": (
        LogisticRegression(max_iter=2000),
        {"C": [0.01, 0.1, 1, 10], "class_weight": [None, "balanced"]},
    ),
    "decision_tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {"max_depth": [2, 3, 5, 8], "min_samples_leaf": [20, 50, 100], "class_weight": [None, "balanced"]},
    ),
    "random_forest": (
        RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE),
        {"max_depth": [3, 5, 8], "min_samples_leaf": [20, 50], "max_features": ["sqrt", 0.5],
         "class_weight": [None, "balanced"]},
    ),
    "svm": (
        SVC(kernel="rbf"),
        {"C": [0.1, 1, 10], "gamma": ["scale", 0.01], "class_weight": [None, "balanced"]},
    ),
}

### 4.3 Helper: build a model pipeline

Features are on very different scales (returns are around 0.01, headline counts can be 50). Logistic regression and SVM work much better when every feature is **standardized** (mean 0, standard deviation 1).

We put the scaler and the model together in a scikit-learn **Pipeline**. The scaler then learns its mean and standard deviation only from the data the pipeline is trained on, never from validation or test data. This is another way of preventing leakage.

In [ ]:
def make_pipeline(model, settings):
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", clone(model).set_params(**settings)),   # clone = a fresh, untrained copy
    ])

### 4.4 Helper: evaluate a model

Returns all the metrics we report:

* **accuracy:** share of days predicted correctly;
* **balanced accuracy:** average of the accuracy on up days and on down days (not fooled by class imbalance);
* **precision:** when the model says "up", how often is it right?
* **recall:** of all the real up days, how many did the model catch?
* **F1:** a balance of precision and recall; **macro F1** averages F1 over both classes;
* **ROC-AUC:** how well the model *ranks* days from most to least likely to go up (0.5 = random guessing).

For ROC-AUC we need a score, not just a yes/no answer. Logistic regression and SVM give a `decision_function`; trees give `predict_proba`.

In [ ]:
def evaluate(model, X, y):
    predictions = model.predict(X)
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
    else:
        scores = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, predictions),
        "balanced_accuracy": balanced_accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
        "f1_macro": f1_score(y, predictions, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y, scores),
    }

### 4.5 Helper: pick the best settings on the validation set

For one model and one feature set, this tries every combination in the grid:

1. train on the **training** period;
2. score it on the **validation** period with **macro F1**;
3. keep the best settings.

Why macro F1 and not accuracy? If 53% of days are up days, a model that always says "up" gets 53% accuracy without learning anything. Macro F1 punishes that, because the F1 for "down" would be 0.

In [ ]:
def tune_on_validation(model, grid, columns):
    best_settings, best_score = None, -1
    for values in itertools.product(*grid.values()):
        settings = dict(zip(grid.keys(), values))
        candidate = make_pipeline(model, settings).fit(train[columns], train["target"])
        score = f1_score(val["target"], candidate.predict(val[columns]), average="macro")
        if score > best_score:
            best_settings, best_score = settings, score
    return best_settings, best_score

### 4.6 Baseline: always guess the most common direction

Before training real models we need a reference point. The **majority-class baseline** always predicts whichever direction was most common in the training data. Any model worth using has to beat this.

In [ ]:
results = []
confusions = {}
test_predictions = {}

baseline = DummyClassifier(strategy="most_frequent").fit(train_val[MARKET_FEATURES], train_val["target"])
baseline_pred = baseline.predict(test[MARKET_FEATURES])

results.append({
    "features": "baseline", "model": "majority_class",
    "accuracy": accuracy_score(test["target"], baseline_pred),
    "balanced_accuracy": balanced_accuracy_score(test["target"], baseline_pred),
    "precision": precision_score(test["target"], baseline_pred, zero_division=0),
    "recall": recall_score(test["target"], baseline_pred, zero_division=0),
    "f1": f1_score(test["target"], baseline_pred, zero_division=0),
    "f1_macro": f1_score(test["target"], baseline_pred, average="macro", zero_division=0),
    "roc_auc": 0.5,       # a constant guess can't rank days, so AUC is 0.5 by definition
})
confusions[("baseline", "majority_class")] = confusion_matrix(test["target"], baseline_pred, labels=[0, 1])
print(f"Baseline always predicts: {'up' if baseline_pred[0] == 1 else 'down'}")
print(f"Baseline test accuracy: {results[0]['accuracy']:.3f}")

### 4.7 Train all models

For each feature set and each model:

1. **tune** on the validation period (4.5);
2. **retrain** the chosen settings on training + validation together, since more data usually means a better model;
3. **test once** on the test period, which no step before this has touched.

This takes a minute or two, mostly because of the random forest and SVM grids.

In [ ]:
final_models = {}

for set_name, columns in FEATURE_SETS.items():
    for model_name, (model, grid) in MODELS.items():
        best_settings, validation_score = tune_on_validation(model, grid, columns)

        final_model = make_pipeline(model, best_settings).fit(train_val[columns], train_val["target"])
        final_models[(set_name, model_name)] = final_model

        test_pred = final_model.predict(test[columns])
        test_predictions[(set_name, model_name)] = test_pred
        confusions[(set_name, model_name)] = confusion_matrix(test["target"], test_pred, labels=[0, 1])

        results.append({"features": set_name, "model": model_name,
                        **evaluate(final_model, test[columns], test["target"]),
                        "validation_f1_macro": validation_score, "best_settings": best_settings})
        print(f"finished {model_name:20s} with {set_name}")

results = pd.DataFrame(results)
results.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)

### 4.8 Exploratory k-means clustering

So far everything has been **supervised** (we told the models the answer during training). Here we try an **unsupervised** method: k-means groups stock-days that look alike, **without** being told the label. Afterwards we check whether some groups are followed by more up days than others.

We cluster on five features that describe a day: today's return, recent volatility, volume change, net sentiment, and amount of news. We fit only on the **training** period.

#### 4.8.1 Choose the number of clusters

We try k = 2 to 6 and compute the **silhouette score** for each. It measures how well-separated the clusters are, from −1 (bad) to 1 (very clear clusters). We keep the k with the highest score.

In [ ]:
CLUSTER_FEATURES = ["ret_1", "vol_5", "volume_chg", "sent_net_mean", "news_count_log"]
X_cluster = StandardScaler().fit_transform(train[CLUSTER_FEATURES])

silhouette_by_k = {}
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_cluster)
    silhouette_by_k[k] = silhouette_score(X_cluster, labels, sample_size=min(3000, len(X_cluster)),
                                          random_state=RANDOM_STATE)

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
pd.Series(silhouette_by_k, name="silhouette").plot(marker="o", figsize=(5, 3), title="Silhouette score by k")
plt.xlabel("k (number of clusters)")
plt.show()
print("Best k:", best_k)

#### 4.8.2 Describe each cluster

We fit k-means with the best k and show the **average** of each feature per cluster, plus the share of days followed by an up day. Values are in original units (e.g. `ret_1` = 0.02 means +2% that day). This tells us what kind of day each cluster represents.

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE).fit(X_cluster)
clustered = train.assign(cluster=kmeans.labels_)

cluster_profile = clustered.groupby("cluster")[CLUSTER_FEATURES + ["target"]].mean()
cluster_profile = cluster_profile.rename(columns={"target": "share_up_next_day"})
cluster_profile.insert(0, "days", clustered["cluster"].value_counts().sort_index())
cluster_profile.to_csv(OUTPUT_DIR / "kmeans_profile.csv")
cluster_profile

#### 4.8.3 Compare clusters with the up/down label

A cross-tabulation: for each cluster, what share of days was followed by a down day vs. an up day. If a cluster is far from the overall up-day rate, that kind of day might be worth a closer look.

In [ ]:
cluster_vs_label = pd.crosstab(clustered["cluster"], clustered["target"].map({0: "down next day", 1: "up next day"}),
                               normalize="index")
cluster_vs_label.plot.bar(stacked=True, figsize=(7, 3.5), color=["indianred", "seagreen"],
                          title="Next-day direction by cluster")
plt.axhline(train["target"].mean(), color="black", linestyle="--", linewidth=0.8)
plt.legend(loc="lower right", fontsize=8)
plt.xticks(rotation=0)
plt.show()
cluster_vs_label

---
# Phase 5: Evaluation

### 5.1 All test-set results

One row per model and feature set. The columns are explained in 4.4. `validation_f1_macro` is the score used to choose the settings; the other numbers are all from the **test** period.

In [ ]:
results.drop(columns=["best_settings"])

### 5.2 Market only vs. market + sentiment

The key table for our research question. For each model we show the test score with and without sentiment, and the **difference** (positive = sentiment helped).

In [ ]:
metrics_to_compare = ["accuracy", "balanced_accuracy", "f1_macro", "roc_auc"]
comparison = results[results["features"] != "baseline"].pivot(index="model", columns="features",
                                                              values=metrics_to_compare)
for metric in metrics_to_compare:
    comparison[(metric, "difference")] = (comparison[(metric, "market_plus_sentiment")]
                                          - comparison[(metric, "market_only")])
comparison = comparison.sort_index(axis=1)
comparison.to_csv(OUTPUT_DIR / "market_vs_sentiment.csv")
comparison

### 5.3 Chart: accuracy and ROC-AUC

The same comparison as a chart, with the baseline as a dashed line. Bars below the dashed line mean the model did worse than just guessing the most common direction.

In [ ]:
model_rows = results[results["features"] != "baseline"]
baseline_row = results[results["features"] == "baseline"].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric in zip(axes, ["accuracy", "roc_auc"]):
    sns.barplot(data=model_rows, x="model", y=metric, hue="features", ax=ax)
    ax.axhline(baseline_row[metric], linestyle="--", color="grey", label="baseline")
    ax.set_ylim(0.4, max(0.7, model_rows[metric].max() + 0.05))
    ax.set_title(f"Test {metric}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=130)
plt.show()

### 5.4 Confusion matrices

Each matrix shows, for the test period, how many down days and up days were predicted correctly and incorrectly:

|  | predicted down | predicted up |
|---|---|---|
| **actually down** | correct | false alarm |
| **actually up** | missed | correct |

A model that only ever predicts one direction shows up here as a matrix with an empty column.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
for ax, (set_name, model_name) in zip(axes.flat, confusions):
    sns.heatmap(confusions[(set_name, model_name)], annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["down", "up"], yticklabels=["down", "up"])
    ax.set_title(f"{model_name}\n({set_name})", fontsize=9)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrices.png", dpi=130)
plt.show()

### 5.5 Which features did the random forest rely on?

A random forest can tell us how much each feature helped its trees make decisions (**feature importance**). Sentiment features are shown in orange. If they're near the top, the model found them useful; if they're all at the bottom, it mostly ignored them.

In [ ]:
forest = final_models[("market_plus_sentiment", "random_forest")].named_steps["model"]
importance = pd.Series(forest.feature_importances_, index=MARKET_FEATURES + SENTIMENT_FEATURES).sort_values()

colors = ["darkorange" if name in SENTIMENT_FEATURES else "steelblue" for name in importance.index]
importance.plot.barh(figsize=(7, 7), color=colors, title="Random forest feature importance (orange = sentiment)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=130)
plt.show()

### 5.6 Is the improvement real or just luck?

With a few hundred test days, a difference of 1–2 percentage points in accuracy can easily happen by chance. **McNemar's test** checks this. For each model it looks only at the test days where the two versions *disagree*:

* `b` = days the **market-only** model got right and the sentiment model got wrong;
* `c` = days the **sentiment** model got right and the market-only model got wrong.

If sentiment doesn't matter, `b` and `c` should be about equal. The **p-value** is the chance of seeing a gap this large if there were no real difference. The usual cut-off is p < 0.05.

In [ ]:
y_test = test["target"].to_numpy()
significance = []

for model_name in MODELS:
    market_right = test_predictions[("market_only", model_name)] == y_test
    sentiment_right = test_predictions[("market_plus_sentiment", model_name)] == y_test
    b = int(np.sum(market_right & ~sentiment_right))
    c = int(np.sum(~market_right & sentiment_right))
    p_value = binomtest(c, b + c, 0.5).pvalue if b + c > 0 else 1.0
    significance.append({"model": model_name, "only market-only right (b)": b,
                         "only sentiment right (c)": c, "p_value": p_value,
                         "significant at 0.05": p_value < 0.05})

significance = pd.DataFrame(significance).set_index("model")
significance

### 5.7 Checking our success criteria

Here we go back to the criteria from **Phase 1.4** and check each one automatically.

In [ ]:
sentiment_rows = results[results["features"] == "market_plus_sentiment"]
beats_baseline = (sentiment_rows["accuracy"] > baseline_row["accuracy"]).sum()
improves_accuracy = (comparison[("accuracy", "difference")] > 0).sum()
improves_auc = (comparison[("roc_auc", "difference")] > 0).sum()
significant = significance["significant at 0.05"].sum()
n_models = len(MODELS)

print(f"Data: {DATA_SOURCE}   |   Sentiment scorer: {scorer_used}\n")
print(f"1. Sentiment models that beat the baseline accuracy:       {beats_baseline} of {n_models}")
print(f"2. Models where sentiment improved test accuracy:          {improves_accuracy} of {n_models}")
print(f"   Models where sentiment improved test ROC-AUC:           {improves_auc} of {n_models}")
print(f"3. Models with a statistically significant difference:     {significant} of {n_models}")

all_met = beats_baseline > n_models / 2 and improves_accuracy > n_models / 2 and significant > n_models / 2
print("\nConclusion:", "sentiment helps (all criteria met)" if all_met else "not enough evidence that sentiment helps")
if DATA_SOURCE == "synthetic":
    print("NOTE: this is synthetic data - do not report these results.")

### 5.8 Discussion

*Write this after running on the real data.* Questions to answer:

* Did sentiment help most models, or only one? How big was the gain compared to the baseline?
* Was the gain statistically significant (5.6)?
* Which sentiment features did the random forest use (5.5)?
* Did any k-means cluster have a clearly higher or lower share of up days (4.8.3)? What kind of day was it?
* Do the confusion matrices show any model that just predicts one direction?

### 5.9 Limitations

* **Coverage and matching:** GDELT gives headlines, not full articles, and some company mentions are missed or wrongly matched even after filtering.
* **Timing:** GDELT's timestamp is when GDELT *found* the article, which can be later than when it was published.
* **Noisy sentiment:** FinBERT scores each headline on its own and doesn't know whether the news was already expected.
* **Small scope:** five large tech stocks over three years, so results may not hold for other stocks or periods.
* **Daily data:** much news is priced in within minutes, which daily prices can't capture.
* **No trading costs:** this is a classification study, not a trading strategy.

---
# Phase 6: Deployment

For a course project, "deployment" means delivering the results so others can use and check them: the report, the presentation, and a notebook anyone can rerun.

### 6.1 Save everything

We save the final modeling table and list all the files in `outputs/`. The charts are ready to drop into the report and slides. To download them from Colab, open the folder icon on the left and right-click a file → *Download*.

In [ ]:
data.to_csv(OUTPUT_DIR / "model_dataset.csv", index=False)
headlines.to_csv(OUTPUT_DIR / "headlines_scored.csv", index=False)

for file in sorted(OUTPUT_DIR.iterdir()):
    print(f"{file.name:30s} {file.stat().st_size / 1024:8.1f} KB")

### 6.2 How to reproduce the results

1. Open this notebook in Google Colab.
2. Set `DATA_SOURCE = "online"` (or `"upload"` with the saved CSV files) and keep `USE_FINBERT = True`.
3. Run all cells. `RANDOM_STATE = 42` makes the model results repeatable.
4. Downloaded data is saved in `data/raw/`. Keep those files (for example, copy them to Google Drive) so later runs use exactly the same data, since GDELT's index can change over time.

### 6.3 Conclusion

*Write after running on real data:* answer the research question in one or two sentences, based on 5.7.

### 6.4 Future work

* Use the full article text instead of just headlines.
* Add more companies and sectors, and a longer time period.
* Use intraday (minute-level) prices to see how quickly news gets priced in.
* Try sequence models (e.g. LSTMs) that look at several days of news together.
* Use walk-forward (rolling) evaluation to check that results hold across different market periods.

### 6.5 References and dataset citations

* **GDELT Project.** GDELT DOC 2.0 API. https://www.gdeltproject.org/. Leetaru, K. & Schrodt, P. (2013). *GDELT: Global data on events, location, and tone, 1979–2012.* ISA Annual Convention. Terms of use: https://www.gdeltproject.org/about.html#termsofuse
* **Stooq.** Historical daily stock quotes. https://stooq.com/ (check the terms of use on the site).
* **FinBERT:** Araci, D. (2019). *FinBERT: Financial Sentiment Analysis with Pre-trained Language Models.* arXiv:1908.10063. Model: https://huggingface.co/ProsusAI/finbert
* Chapman, P. et al. (2000). *CRISP-DM 1.0: Step-by-step data mining guide.* SPSS Inc.
* Pedregosa, F. et al. (2011). *Scikit-learn: Machine Learning in Python.* JMLR 12, 2825–2830.
* Wolf, T. et al. (2020). *Transformers: State-of-the-Art Natural Language Processing.* EMNLP: System Demonstrations.
* McNemar, Q. (1947). Note on the sampling error of the difference between correlated proportions or percentages. *Psychometrika* 12(2), 153–157.